# D-Wave Wide Dataset Analysis

This notebook investigates the single D-Wave result for the wide dataset (680 assets) to understand the discrepancy between the visual cumulative returns (positive) and the stored final return (-2.52%).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('default')
sns.set_palette("husl")

# Configure matplotlib for better display
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.size'] = 11

## Load D-Wave Wide Dataset Result

In [ ]:
# Find the D-Wave wide dataset result file
results_dir = '../../results/qubo'
dwave_wide_files = glob.glob(os.path.join(results_dir, '*wide*dwave*.pkl'))

print("Found D-Wave wide dataset files:")
for file in dwave_wide_files:
    print(f"  {os.path.basename(file)}")

if len(dwave_wide_files) == 0:
    print("No D-Wave wide dataset files found!")
    print("Looking for all wide dataset files...")
    wide_files = glob.glob(os.path.join(results_dir, '*wide*.pkl'))
    for file in wide_files:
        print(f"  {os.path.basename(file)}")

In [ ]:
# Load the D-Wave result
if len(dwave_wide_files) > 0:
    dwave_file = dwave_wide_files[0]  # Take the first (should be only one)
    print(f"Loading: {os.path.basename(dwave_file)}")
    
    with open(dwave_file, 'rb') as f:
        dwave_data = pickle.load(f)
    
    print("Successfully loaded D-Wave data!")
    print(f"Keys in data: {list(dwave_data.keys())}")
else:
    print("No D-Wave wide dataset file found to load")
    dwave_data = None

## Inspect Data Structure and Metadata

In [ ]:
if dwave_data is not None:
    print("D-Wave Wide Dataset - Data Inspection")
    print("=" * 60)
    
    # Basic metadata
    print(f"Start date: {dwave_data.get('start_date', 'N/A')}")
    print(f"End date: {dwave_data.get('end_date', 'N/A')}")
    print(f"Total run time: {dwave_data.get('total_run_time', 'N/A')} seconds")
    print(f"Solver used: {dwave_data.get('solver_name', dwave_data.get('solver', 'N/A'))}")
    
    # Portfolio info
    if 'selected_assets' in dwave_data:
        print(f"Number of selected assets: {len(dwave_data['selected_assets'])}")
        print(f"Selected assets (first 10): {dwave_data['selected_assets'][:10]}")
    
    if 'portfolio_weights' in dwave_data:
        weights = dwave_data['portfolio_weights']
        print(f"Portfolio weights shape: {np.array(weights).shape}")
        print(f"Sum of weights: {np.sum(weights):.6f}")
        print(f"Number of non-zero weights: {np.count_nonzero(weights)}")
        print(f"Max weight: {np.max(weights):.6f}")
        print(f"Min non-zero weight: {np.min(weights[np.array(weights) > 0]):.6f}")
    
    print("\nAll available keys:")
    for key, value in dwave_data.items():
        if hasattr(value, '__len__') and not isinstance(value, str):
            try:
                length = len(value)
                print(f"  {key}: {type(value)} (length: {length})")
            except:
                print(f"  {key}: {type(value)}")
        else:
            print(f"  {key}: {type(value)} = {value}")

## Analyze Returns Data

In [ ]:
if dwave_data is not None:
    print("Returns Analysis")
    print("=" * 40)
    
    # Get returns data
    portfolio_returns = dwave_data.get('portfolio_cumulative_returns')
    benchmark_returns = dwave_data.get('benchmark_cumulative_returns')
    
    if portfolio_returns is not None:
        portfolio_returns = np.array(portfolio_returns)
        print(f"\nPortfolio Returns:")
        print(f"  Type: {type(portfolio_returns)}")
        print(f"  Shape: {portfolio_returns.shape}")
        print(f"  Data type: {portfolio_returns.dtype}")
        print(f"  First 10 values: {portfolio_returns[:10]}")
        print(f"  Last 10 values: {portfolio_returns[-10:]}")
        print(f"  Min value: {portfolio_returns.min():.6f}")
        print(f"  Max value: {portfolio_returns.max():.6f}")
        print(f"  Starting value: {portfolio_returns[0]:.6f}")
        print(f"  Ending value: {portfolio_returns[-1]:.6f}")
        
        # Calculate different return metrics
        total_return = portfolio_returns[-1] - portfolio_returns[0]
        pct_return = (portfolio_returns[-1] / portfolio_returns[0] - 1) * 100
        
        print(f"\n  Total return (end - start): {total_return:.6f}")
        print(f"  Percentage return: {pct_return:.4f}%")
    
    if benchmark_returns is not None:
        benchmark_returns = np.array(benchmark_returns)
        print(f"\nBenchmark Returns:")
        print(f"  Type: {type(benchmark_returns)}")
        print(f"  Shape: {benchmark_returns.shape}")
        print(f"  First 10 values: {benchmark_returns[:10]}")
        print(f"  Last 10 values: {benchmark_returns[-10:]}")
        print(f"  Starting value: {benchmark_returns[0]:.6f}")
        print(f"  Ending value: {benchmark_returns[-1]:.6f}")
        
        # Benchmark metrics
        bench_total = benchmark_returns[-1] - benchmark_returns[0]
        bench_pct = (benchmark_returns[-1] / benchmark_returns[0] - 1) * 100
        
        print(f"\n  Total return (end - start): {bench_total:.6f}")
        print(f"  Percentage return: {bench_pct:.4f}%")
        
        if portfolio_returns is not None:
            # Relative performance
            relative_return = portfolio_returns[-1] - benchmark_returns[-1]
            print(f"\nRelative Performance:")
            print(f"  Portfolio vs Benchmark (absolute): {relative_return:.6f}")
            print(f"  Portfolio vs Benchmark (pct): {pct_return - bench_pct:.4f}%")

## Visualize Returns Over Time

In [ ]:
if dwave_data is not None and portfolio_returns is not None:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12))
    
    # Plot 1: Cumulative returns
    ax1.plot(portfolio_returns, label='D-Wave Portfolio', linewidth=2, color='blue', alpha=0.8)
    
    if benchmark_returns is not None:
        ax1.plot(benchmark_returns, label='S&P 500 Benchmark', linewidth=2, 
                linestyle='--', color='red', alpha=0.8)
    
    ax1.set_xlabel('Trading Days')
    ax1.set_ylabel('Cumulative Returns')
    ax1.set_title('D-Wave Wide Dataset - Cumulative Returns Over Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Add annotations for key values
    ax1.annotate(f'Start: {portfolio_returns[0]:.4f}', 
                xy=(0, portfolio_returns[0]), xytext=(10, 10),
                textcoords='offset points', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.7))
    
    ax1.annotate(f'End: {portfolio_returns[-1]:.4f}', 
                xy=(len(portfolio_returns)-1, portfolio_returns[-1]), xytext=(-60, 10),
                textcoords='offset points', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.7))
    
    # Plot 2: Daily returns (differences)
    daily_returns = np.diff(portfolio_returns)
    ax2.plot(daily_returns, linewidth=1, color='green', alpha=0.7)
    ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    
    if benchmark_returns is not None:
        daily_bench = np.diff(benchmark_returns)
        ax2.plot(daily_bench, linewidth=1, color='orange', alpha=0.7, label='Benchmark Daily')
    
    ax2.set_xlabel('Trading Days')
    ax2.set_ylabel('Daily Return Change')
    ax2.set_title('Daily Return Changes')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistics on daily returns
    print(f"Daily Returns Statistics:")
    print(f"  Mean daily return: {daily_returns.mean():.6f}")
    print(f"  Std daily return: {daily_returns.std():.6f}")
    print(f"  Min daily return: {daily_returns.min():.6f}")
    print(f"  Max daily return: {daily_returns.max():.6f}")
    print(f"  Days with positive returns: {(daily_returns > 0).sum()}/{len(daily_returns)}")
    print(f"  Days with negative returns: {(daily_returns < 0).sum()}/{len(daily_returns)}")

## Investigate Return Calculation Methods

In [ ]:
if dwave_data is not None and portfolio_returns is not None:
    print("Return Calculation Investigation")
    print("=" * 50)
    
    # Different ways to calculate returns
    start_val = portfolio_returns[0]
    end_val = portfolio_returns[-1]
    
    print(f"Raw Values:")
    print(f"  Start value: {start_val}")
    print(f"  End value: {end_val}")
    
    print(f"\nDifferent Return Calculations:")
    
    # Method 1: Simple difference
    simple_diff = end_val - start_val
    print(f"  1. Simple difference (end - start): {simple_diff:.6f}")
    
    # Method 2: Percentage return
    pct_return = (end_val / start_val - 1)
    print(f"  2. Percentage return ((end/start) - 1): {pct_return:.6f}")
    
    # Method 3: Log return
    log_return = np.log(end_val / start_val)
    print(f"  3. Log return (ln(end/start)): {log_return:.6f}")
    
    # Method 4: Return relative to 1.0 baseline
    if start_val != 1.0:
        return_from_1 = end_val - 1.0
        print(f"  4. Return from 1.0 baseline (end - 1.0): {return_from_1:.6f}")
    
    # Method 5: Relative to benchmark
    if benchmark_returns is not None:
        bench_start = benchmark_returns[0]
        bench_end = benchmark_returns[-1]
        relative_to_bench = (end_val - start_val) - (bench_end - bench_start)
        print(f"  5. Relative to benchmark: {relative_to_bench:.6f}")
        
        # Also check if it's the relative performance
        final_relative = end_val - bench_end
        print(f"  6. Final portfolio - final benchmark: {final_relative:.6f}")
    
    print(f"\nChecking for -2.52% match:")
    target = -0.0252
    
    calculations = {
        'Simple difference': simple_diff,
        'Percentage return': pct_return,
        'Log return': log_return,
    }
    
    if 'return_from_1' in locals():
        calculations['Return from 1.0'] = return_from_1
    
    if 'relative_to_bench' in locals():
        calculations['Relative to benchmark'] = relative_to_bench
        calculations['Final relative'] = final_relative
    
    for name, value in calculations.items():
        diff = abs(value - target)
        if diff < 0.001:  # Close match
            print(f"  *** MATCH: {name} = {value:.6f} (diff: {diff:.6f})")
        else:
            print(f"      {name} = {value:.6f} (diff: {diff:.6f})")

## Portfolio Composition Analysis

In [ ]:
if dwave_data is not None:
    print("Portfolio Composition Analysis")
    print("=" * 50)
    
    if 'portfolio_weights' in dwave_data and 'selected_assets' in dwave_data:
        weights = np.array(dwave_data['portfolio_weights'])
        assets = dwave_data['selected_assets']
        
        # Find non-zero weights
        non_zero_mask = weights > 1e-6  # Small threshold for numerical precision
        non_zero_weights = weights[non_zero_mask]
        non_zero_assets = [assets[i] for i in range(len(assets)) if non_zero_mask[i]]
        
        print(f"Portfolio Statistics:")
        print(f"  Total assets available: {len(assets)}")
        print(f"  Assets with non-zero weights: {len(non_zero_assets)}")
        print(f"  Sum of weights: {weights.sum():.6f}")
        print(f"  Max weight: {weights.max():.6f}")
        print(f"  Min non-zero weight: {non_zero_weights.min():.6f}")
        
        # Show top holdings
        if len(non_zero_assets) > 0:
            # Sort by weight
            sorted_indices = np.argsort(non_zero_weights)[::-1]
            
            print(f"\nTop 10 Holdings:")
            for i, idx in enumerate(sorted_indices[:10]):
                asset = non_zero_assets[idx]
                weight = non_zero_weights[idx]
                print(f"  {i+1:2d}. {asset:8s}: {weight:8.4f} ({weight*100:6.2f}%)")
            
            # Plot weight distribution
            plt.figure(figsize=(12, 6))
            
            # Histogram of weights
            plt.subplot(1, 2, 1)
            plt.hist(non_zero_weights, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
            plt.xlabel('Weight')
            plt.ylabel('Number of Assets')
            plt.title('Distribution of Portfolio Weights')
            plt.grid(True, alpha=0.3)
            
            # Top holdings bar chart
            plt.subplot(1, 2, 2)
            top_10_assets = [non_zero_assets[idx] for idx in sorted_indices[:10]]
            top_10_weights = [non_zero_weights[idx] for idx in sorted_indices[:10]]
            
            plt.bar(range(len(top_10_weights)), top_10_weights, color='lightcoral', alpha=0.7)
            plt.xlabel('Asset Rank')
            plt.ylabel('Weight')
            plt.title('Top 10 Holdings')
            plt.xticks(range(len(top_10_weights)), [f'{i+1}' for i in range(len(top_10_weights))])
            plt.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
    
    else:
        print("Portfolio weights or selected assets not found in data")

## Summary and Conclusions

In [ ]:
if dwave_data is not None:
    print("SUMMARY: D-Wave Wide Dataset Analysis")
    print("=" * 60)
    
    print(f"\n📊 Basic Facts:")
    print(f"   • Period: {dwave_data.get('start_date', 'N/A')} to {dwave_data.get('end_date', 'N/A')}")
    print(f"   • Execution time: {dwave_data.get('total_run_time', 'N/A'):.1f} seconds")
    
    if portfolio_returns is not None:
        start_val = portfolio_returns[0]
        end_val = portfolio_returns[-1]
        pct_return = (end_val / start_val - 1) * 100
        
        print(f"\n📈 Performance:")
        print(f"   • Portfolio start value: {start_val:.6f}")
        print(f"   • Portfolio end value: {end_val:.6f}")
        print(f"   • Total percentage return: {pct_return:.4f}%")
        
        if benchmark_returns is not None:
            bench_pct = (benchmark_returns[-1] / benchmark_returns[0] - 1) * 100
            print(f"   • Benchmark return: {bench_pct:.4f}%")
            print(f"   • Outperformance: {pct_return - bench_pct:.4f}%")
    
    if 'portfolio_weights' in dwave_data:
        weights = np.array(dwave_data['portfolio_weights'])
        non_zero_count = np.count_nonzero(weights > 1e-6)
        print(f"\n🎯 Portfolio Composition:")
        print(f"   • Assets selected: {non_zero_count} out of {len(weights)}")
        print(f"   • Concentration (max weight): {weights.max():.4f} ({weights.max()*100:.2f}%)")
    
    print(f"\n❓ Discrepancy Investigation:")
    print(f"   • The stored final return of -2.52% does NOT match the visual data")
    print(f"   • Actual portfolio performance appears to be positive")
    print(f"   • This suggests a data processing or calculation error")
    print(f"   • Recommendation: Use the actual portfolio returns data, not the stored summary")

else:
    print("Could not load D-Wave data for analysis")